# VN30 — Pipeline train (định nghĩa) & báo cáo từ bundle

Notebook gồm hai phần:

- **Phần A (ô dưới):** định nghĩa **từng bước train** + tên **method** trong `stock_analysis_yfinance.py` — *không chạy train trong notebook này*.
- **Phần B (ô code):** đọc `analysis_bundle.joblib` đã train sẵn + vẽ chart (cần upload cùng `stock_analysis_yfinance.py`, `bundle_chart_report.py`).

**Upload Colab:** `colab.ipynb` + `analysis_bundle.joblib` + `stock_analysis_yfinance.py` + `bundle_chart_report.py` vào cùng thư mục làm việc (vd. `/content`).

**Thứ tự trên Colab:** chạy ô **pip** đầu tiên → **Runtime → Restart session** → rồi chạy Phần B. Colab giữ pandas cũ trong RAM nếu không restart → lỗi `StringDtype` / `joblib.load`.


## Phần A — Định nghĩa pipeline train & cách chọn / so sánh mô hình

> Mô tả khớp `stock_analysis_yfinance.py` (phiên bản repo hiện tại). **Train thật** chạy: `python stock_analysis_yfinance.py` hoặc `StockAnalysisYFinance().run_full_pipeline(period="5y", refresh_cache=False)`.

### A.0 Chuỗi tổng (`run_full_pipeline`)

| Bước | Method | Vai trò |
|------|--------|---------|
| 1 | `collect_stock_data_yfinance` | Tải VN30 (Yahoo), `period` vd. `5y`, cache parquet, chọn **Top mã**. |
| 2 | `preprocess_data` | Missing / outlier. |
| 3 | `run_eda` | EDA + ảnh `artifacts/eda/`. |
| 4 | `calculate_technical_indicators` | EMA, RSI, MACD, Bollinger, OBV, VWAP, … |
| 5 | `add_hybrid_econometric_features` | Đặc trưng hybrid / joint (tuỳ cấu hình). |
| 6 | `prepare_features` | Bảng mẫu + nhãn `target_class`; có thể có **meta-labeling** (`meta_label`, `_apply_meta_labeling`). |
| 7 | `train_model` | ML + econometric + benchmark + rolling. |
| 8 | `save_model` | `analysis_bundle.joblib`, … |

---

### A.1 Nhánh ML: `train_ml_models` vs `_train_ml_models_meta`

- Nếu **`TargetConfig.use_meta_labeling`** là **True** và đã có cột `meta_label`: gọi **`_train_ml_models_meta`** (nhãn **nhị phân** “meta”, kiểu AFML triple-barrier / primary side).  
- Ngược lại: gọi **`train_ml_models`** — nhãn **`target_class`** đa lớp (**TANG / SIDEWAY / GIAM**).

Dưới đây là **`train_ml_models`** (đa lớp); meta chỉ khác **nhãn + một số factory XGB/LGB** (binary).

---

### A.2 Các mô hình ML trong code — có gì, khi nào chạy được?

Tất cả đều được khởi tạo trong dict **`model_factories`** (hyperparameter cố định trong code):

| Mô hình | Tham số chính (tóm tắt) | Điều kiện xuất hiện |
|---------|-------------------------|---------------------|
| **RandomForest** | `n_estimators=200`, `max_depth=10`, `class_weight="balanced"` | Luôn có. |
| **LogisticRegression** | Pipeline `StandardScaler` + `LogisticRegression`, `max_iter=2500`, **multinomial** / `lbfgs` (đa lớp) | Luôn có. |
| **HistGradientBoosting** | `max_iter=200`, `max_depth=6`, `lr=0.05` | Luôn có. |
| **GradientBoosting** | `n_estimators=150`, `max_depth=4`, `subsample=0.9` | Luôn có. |
| **XGBoost** | `XGBClassifier`, `multi:softprob`, `mlogloss`, 200 cây, … | Chỉ khi import **`xgboost`** thành công. |
| **LightGBM** | `LGBMClassifier`, multiclass, … | Chỉ khi import **`lightgbm`** thành công. |
| **LSTM** | `LSTMClassifierWrapper`: 16 unit, 20 epoch, batch 64, … | Chỉ khi có **TensorFlow** (`_get_tensorflow()`). |

**Không có trong bundle** = thư viện chưa cài **hoặc** cặp (model, feature set) **fail** `_run_ml_data_checks` **hoặc** train ném exception → bị **bỏ qua** (warning).

---

### A.3 Chia dữ liệu theo thời gian (train / val / test)

Trên `feature_data` đã sort theo `date`:

- `test_ratio` (trong `TargetConfig`, thường **0.2**) → phần cuối là **test**.  
- Phần **trước test** chia tiếp: `val_ratio` (thường **0.15** của đoạn “pre-test”) → **validation**.  
- Còn lại là **train**.

→ **Không shuffle ngẫu nhiên**; đúng kiểu chuỗi thời gian.

---

### A.4 Bộ feature được so sánh (trong `train_ml_models` chuẩn)

Với mỗi thuật toán, pipeline thử **nhiều không gian đặc trưng** — code hiện tại định nghĩa **`core`** và **`core_seasonal`** (sau khi loại trùng signature):

- **`core`**: `core_feature_cols` + **regime** + **market context** (nếu có trong bảng).  
- **`core_seasonal`**: `core` + **`seasonal_feature_cols`**.

Trên **mỗi** bộ gốc `original_cols`:

1. **`_prune_multicollinearity`** (train) → bỏ cặp tương quan cao.  
2. **`_filter_by_importance`** (train vs val) → giữ cột quan trọng.  
3. **`_run_ml_data_checks`** → nếu **không pass**, **bỏ** hẳn cặp (model, feature_set) đó.

---

### A.5 Hàm `train_candidate` — chạy và so sánh baseline vs đã lọc

Cho **một** thuật toán và **một** bộ cột đã lọc `selected_cols` (và `original_cols` để baseline):

1. **Baseline:** cùng factory, fit trên **train** với **`original_cols`**, đo **val** và **test**.  
2. **Ứng viên:** fit trên **train** với **`selected_cols`**, đo **val** và **test**.  
3. **Refit báo cáo test:** fit lại trên **train ∪ val** rồi đánh giá **test** (cả baseline và ứng viên).

**`baseline_metrics`** = baseline **full original** trên train∪val → test. **`metrics`** = mô hình **sau multicollinearity + importance** trên train∪val → test.

**`delta_vs_baseline`** = chênh **accuracy / F1 / precision** (test) giữa hai.

---

### A.6 Chọn **một bộ feature** cho mỗi tên mô hình (vd. RandomForest)

Trong các `feature_set` đã pass kiểm tra và train thành công:

- Chọn bộ có **`val_f1`** (**F1 weighted trên validation**) **cao nhất** (`selection_metric` = `val_f1`).  
- Kết quả lưu: **`selected_feature_set`** (vd. `core` hoặc `core_seasonal`), **`feature_set_comparison`** (mọi bộ đã thử).

---

### A.7 Chọn **mô hình đại diện** cho `feature_cols` mặc định (toàn project)

Trong các mô hình đã train xong:

- Chọn tên mô hình có **`f1_weighted` trên test** (**`metrics["f1_weighted"]`**) **cao nhất** → gán **`self.feature_cols`** = bộ cột của mô hình đó (dùng cho inference / checklist).

---

### A.8 Bảng `ranking` trong `build_benchmark_summary`

- Mỗi hàng ML = một tên + **`selected_feature_set`** + toàn bộ metric **test**.  
- Thêm hàng **Econometric** (trung bình theo mã) nếu có.  
- Sort **`benchmark_df`** theo **`accuracy`** giảm dần, rồi **`f1_weighted`** giảm dần → thứ tự **`ranking`**.  
- **`top_model`**: dòng đầu sau sort.

**Lưu ý:** “Đứng đầu **ranking**” ưu tiên **accuracy** rồi mới **f1** — **có thể khác** mô hình có **F1 test cao nhất** trong A.7 (A.7 chỉ để chọn `feature_cols`, không phải sort ranking).

---

### A.9 Meta (`_train_ml_models_meta`)

- Nhãn **`meta_label`** (binary); LogReg / XGB / LGB dùng **binary** trong factory.  
- Vẫn chọn feature set theo **`val_f1`**, chọn mô hình tổng thể theo **f1 test**.

---

### A.10 Trong bundle — khớp bảng kết quả của bạn

- **`benchmark_summary["ml_models"]`**: metrics + `selected_feature_set` + `feature_set_comparison` + `delta_vs_baseline`.  
- **`benchmark_summary["ranking"]`**: bảng tổng để so ML vs econometric.  
- **`ml_predictions_store`**: dự báo + `test_frame` trên tập test.

---


In [ ]:
# Colab: nâng pandas (bundle train với pandas 2.2+). Bỏ -U hoặc không restart → dễ lỗi StringDtype khi joblib.load.
%pip install -q -U "pandas>=2.2,<3" "numpy>=1.26,<3" matplotlib seaborn "joblib>=1.3" "scikit-learn>=1.4" xgboost statsmodels arch pyarrow yfinance
# TensorFlow: chỉ khi cần LSTM đầy đủ trong RAM; chart có thể bỏ (patch trong bundle_chart_report)
# %pip install -q tensorflow


## Phần B — Đọc bundle + sinh biểu đồ báo cáo

Chạy các ô code bên dưới (cần `analysis_bundle.joblib` + `stock_analysis_yfinance.py` + `bundle_chart_report.py` trong `ROOT`).

**Bắt buộc:** sau ô `pip` phải **Runtime → Restart session**, rồi mới import/chạy ô dưới (không restart thì vẫn lỗi unpickle dù đã cài pandas mới).

Patch cho phép unpickle bundle **không cần TensorFlow** để vẽ chart (LSTM trong RAM có thể không khôi phục đầy đủ).


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from bundle_chart_report import patch_lstm_unpickle_allow_missing_tf, generate_all_charts
from stock_analysis_yfinance import StockAnalysisYFinance

patch_lstm_unpickle_allow_missing_tf()
model = StockAnalysisYFinance()
ok = model.load_model_bundle()
assert ok, (
    "Không load được analysis_bundle.joblib — đặt file cùng ROOT hoặc chỉnh đường dẫn."
)

OUT = ROOT / "artifacts/colab_charts"
report = generate_all_charts(model, OUT)
print("Saved", len(report["figures"]), "figures")
report["figures"][:5]


## Phần B (tiếp) — Hiển thị biểu đồ đã lưu


In [ ]:
from IPython.display import Image, display

for fp in sorted(Path(report["out_dir"]).glob("*.png"))[:8]:
    display(Image(filename=str(fp)))
